# Windowing + PCA — CC1-Only Design

Input: `data/processed/{cc1_train,cc1_val,cc1_test,drift_single_case1,drift_single_case2,drift_complex_case2}.csv`
(from `clean_and_split.ipynb` — deduplicated, gap-marked, delay/loss rescoped, CC1-train-scaled).

**Scope of this notebook:** sliding-window construction → flatten → PCA. Feeds directly
into `train_vae.ipynb`. Per the Module 3 spec (VAE + PCA for multi-metric correlation,
sliding window 30–60 timesteps), this builds the window/PCA representation the VAE
trains on.

In [1]:
import pandas as pd
import numpy as np
import joblib, os
from sklearn.decomposition import PCA

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
OUT_DIR   = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

FEATURE_COLS = [
    'container_cpu_usage_seconds_rate',
    'container_cpu_system_seconds_rate',
    'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes',
    'container_memory_working_set_bytes',
    'container_memory_rss',
    'container_memory_cache',
]

SPLIT_FILES = {
    'cc1_train':      'cc1_train.csv',
    'cc1_val':        'cc1_val.csv',
    'cc1_test':       'cc1_test.csv',
    'drift_sc1':      'drift_single_case1.csv',
    'drift_sc2':      'drift_single_case2.csv',
    'drift_cc2':      'drift_complex_case2.csv',
}

WINDOW_SIZE   = 30     # 30 x 15s = 7.5 minutes (within the spec's 30-60 timestep range)
STRIDE        = 1      # rolling window, maximizes training samples
PCA_VARIANCE  = 0.99    # retain components explaining 99% of variance (fit on cc1_train only) — see Step 5 note

print('Paths and constants configured.')
print(f'WINDOW_SIZE={WINDOW_SIZE}  STRIDE={STRIDE}  PCA_VARIANCE={PCA_VARIANCE}')

Paths and constants configured.
WINDOW_SIZE=30  STRIDE=1  PCA_VARIANCE=0.99


## Step 1 — Load split files

In [2]:
splits = {}
for name, fname in SPLIT_FILES.items():
    path = os.path.join(DATA_DIR, fname)
    d = pd.read_csv(path, low_memory=False)
    d['is_gap'] = d['is_gap'].astype(bool)
    splits[name] = d
    print(f'  {name:10s}: {len(d):>7,} rows  |  containers: {d["cmdb_id"].nunique()}  |  '
          f'anomalies: {int(d["label"].sum()):,}')

  cc1_train : 156,479 rows  |  containers: 27  |  anomalies: 0
  cc1_val   :  22,356 rows  |  containers: 27  |  anomalies: 0
  cc1_test  :  44,968 rows  |  containers: 27  |  anomalies: 256
  drift_sc1 :  60,480 rows  |  containers: 27  |  anomalies: 76
  drift_sc2 :  38,853 rows  |  containers: 27  |  anomalies: 56
  drift_cc2 :  77,760 rows  |  containers: 27  |  anomalies: 372


## Step 2 — Sliding-window construction (gap-aware)

Per container: slide a window of `WINDOW_SIZE` rows, stride `STRIDE`. A window is
**skipped** if any row in it starts a time gap (`is_gap`) — i.e. the container's
15s cadence was broken somewhere inside that window, so the 30 steps wouldn't be a
true contiguous 7.5-minute sequence. A window is labeled anomaly (`y=1`) if **any**
timestep inside it is anomalous; `failure_type` records the fault type(s) present
(comma-joined if more than one, for reference only — not used in training).

In [3]:
def build_windows(container_df, feature_cols, window_size, stride):
    data    = container_df[feature_cols].values.astype(np.float32)
    labels  = container_df['label'].values
    ftypes  = container_df['failure_type'].values.astype(object)
    is_gap  = container_df['is_gap'].values
    n       = len(data)

    X, y, ft = [], [], []
    for i in range(0, n - window_size + 1, stride):
        if is_gap[i : i + window_size].any():
            continue
        X.append(data[i : i + window_size])
        window_labels = labels[i : i + window_size]
        y.append(int(window_labels.any()))
        w_types = sorted({t for t in ftypes[i : i + window_size] if isinstance(t, str)})
        ft.append(','.join(w_types) if w_types else None)

    if not X:
        return (np.empty((0, window_size, len(feature_cols)), dtype=np.float32),
                np.empty((0,), dtype=np.int64), np.array([], dtype=object))
    return np.stack(X), np.array(y, dtype=np.int64), np.array(ft, dtype=object)


def window_split(df, feature_cols, window_size, stride):
    Xs, ys, fts = [], [], []
    for cmdb_id, g in df.sort_values('timestamp').groupby('cmdb_id'):
        X, y, ft = build_windows(g, feature_cols, window_size, stride)
        if len(X):
            Xs.append(X); ys.append(y); fts.append(ft)
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(fts)

print('Window builder defined.')

Window builder defined.


## Step 3 — Apply windowing to every split

In [4]:
windowed = {}
for name, d in splits.items():
    X, y, ft = window_split(d, FEATURE_COLS, WINDOW_SIZE, STRIDE)
    windowed[name] = {'X': X, 'y': y, 'ft': ft}
    n_anom = int(y.sum())
    print(f'  {name:10s}: {len(d):>7,} rows -> {len(y):>7,} windows  '
          f'({n_anom:,} anomaly windows, {n_anom / max(len(y),1) * 100:.2f}%)')

  cc1_train : 156,479 rows -> 154,198 windows  (0 anomaly windows, 0.00%)
  cc1_val   :  22,356 rows ->  21,573 windows  (0 anomaly windows, 0.00%)
  cc1_test  :  44,968 rows ->  44,185 windows  (256 anomaly windows, 0.58%)
  drift_sc1 :  60,480 rows ->  59,697 windows  (134 anomaly windows, 0.22%)
  drift_sc2 :  38,853 rows ->  38,070 windows  (85 anomaly windows, 0.22%)
  drift_cc2 :  77,760 rows ->  76,977 windows  (720 anomaly windows, 0.94%)


## Step 4 — Flatten windows

`(N, 30, 7) -> (N, 210)` so PCA can operate on each window as a single vector.

In [5]:
for name in windowed:
    X = windowed[name]['X']
    X_flat = X.reshape(len(X), -1)
    windowed[name]['X_flat'] = X_flat
    print(f'  {name:10s}: {X.shape} -> {X_flat.shape}')

  cc1_train : (154198, 30, 7) -> (154198, 210)
  cc1_val   : (21573, 30, 7) -> (21573, 210)
  cc1_test  : (44185, 30, 7) -> (44185, 210)
  drift_sc1 : (59697, 30, 7) -> (59697, 210)
  drift_sc2 : (38070, 30, 7) -> (38070, 210)
  drift_cc2 : (76977, 30, 7) -> (76977, 210)


## Step 5 — PCA (fit on cc1_train only)

`cc1_train` is entirely normal by construction (no leakage risk from anomalies or
other cases). Two decisions here were driven by checking the actual explained-variance
curve first, not by defaulting to convention:

**99% variance, not 95%.** At 95%, only **5** components survive, and CPU-loaded
components (`cpu_system`-dominant) contribute barely ~2% combined variance — right at
the cutoff edge. Since `cpu` is one of only 3 detectable fault types this module
targets (the others being `memory`, `pod-failure`), throwing away that tail right where
CPU signal lives would blunt exactly the anomaly type we need to keep. 99% keeps ~26
components instead, preserving far more of that low-variance-but-informative tail.

**Whitening.** PCA orders components by variance, so PC1/PC2 (memory-dominated, ~93%
of variance) have a much larger raw scale than the CPU-loaded components further down.
A plain reconstruction-MSE VAE naturally weights loss by scale — it would optimize
almost entirely for PC1/PC2 and barely notice deviations in the small-scale
CPU-carrying components, exactly where we need sensitivity. `whiten=True` rescales
every retained component to unit variance, so all of them contribute comparably to
the VAE's reconstruction loss regardless of how much raw variance they originally held.

In [6]:
X_train_flat = windowed['cc1_train']['X_flat']
print(f'Fitting PCA on {len(X_train_flat):,} cc1_train windows (all normal) ...')
print(f'Input dimension: {X_train_flat.shape[1]} (= {WINDOW_SIZE} timesteps x {len(FEATURE_COLS)} features)')

pca = PCA(n_components=PCA_VARIANCE, svd_solver='full', whiten=True, random_state=42)
pca.fit(X_train_flat)
n_components = pca.n_components_
print(f'Components retained for {PCA_VARIANCE*100:.0f}% variance: {n_components}  '
      f'(compression {X_train_flat.shape[1]} -> {n_components})')
print(f'explained_variance_ratio_ (first 10): {np.round(pca.explained_variance_ratio_[:10], 4)}')

for name in windowed:
    windowed[name]['X_pca'] = pca.transform(windowed[name]['X_flat']).astype(np.float32)
    print(f'  {name:10s}: {windowed[name]["X_flat"].shape} -> {windowed[name]["X_pca"].shape}')

print()
print('Whitened cc1_train PCA-space stats (each component should now be ~unit variance):')
print(pd.DataFrame(windowed['cc1_train']['X_pca']).describe().loc[['mean', 'std']].round(3).iloc[:, :10])

Fitting PCA on 154,198 cc1_train windows (all normal) ...
Input dimension: 210 (= 30 timesteps x 7 features)
Components retained for 99% variance: 26  (compression 210 -> 26)
explained_variance_ratio_ (first 10): [0.7511 0.1756 0.0102 0.0101 0.0069 0.0031 0.003  0.0023 0.0023 0.0023]
  cc1_train : (154198, 210) -> (154198, 26)
  cc1_val   : (21573, 210) -> (21573, 26)
  cc1_test  : (44185, 210) -> (44185, 26)
  drift_sc1 : (59697, 210) -> (59697, 26)
  drift_sc2 : (38070, 210) -> (38070, 26)
  drift_cc2 : (76977, 210) -> (76977, 26)

Whitened cc1_train PCA-space stats (each component should now be ~unit variance):
        0    1    2    3    4    5    6    7    8    9
mean  0.0  0.0  0.0  0.0  0.0  0.0  0.0 -0.0  0.0 -0.0
std   1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0


## Step 6 — Save windows + PCA model

In [7]:
for name, d in windowed.items():
    np.save(os.path.join(OUT_DIR, f'X_{name}.npy'),  d['X_pca'])
    np.save(os.path.join(OUT_DIR, f'y_{name}.npy'),  d['y'])
    np.save(os.path.join(OUT_DIR, f'ft_{name}.npy'), d['ft'])
    print(f'  saved X/y/ft_{name}.npy  ({len(d["y"]):,} windows)')

pca_path = os.path.join(MODEL_DIR, 'cc1_pca.pkl')
joblib.dump({
    'pca': pca,
    'feature_cols': FEATURE_COLS,
    'window_size': WINDOW_SIZE,
    'stride': STRIDE,
    'n_components': n_components,
}, pca_path)
print(f'\n  PCA model saved -> {pca_path}')

  saved X/y/ft_cc1_train.npy  (154,198 windows)
  saved X/y/ft_cc1_val.npy  (21,573 windows)
  saved X/y/ft_cc1_test.npy  (44,185 windows)
  saved X/y/ft_drift_sc1.npy  (59,697 windows)
  saved X/y/ft_drift_sc2.npy  (38,070 windows)
  saved X/y/ft_drift_cc2.npy  (76,977 windows)

  PCA model saved -> c:\Users\jthar\Documents\Claude\Projects\module3\models\cc1_pca.pkl


## Step 7 — Final verification

In [8]:
print('=== FINAL VERIFICATION ===\n')
for name in windowed:
    X = np.load(os.path.join(OUT_DIR, f'X_{name}.npy'))
    y = np.load(os.path.join(OUT_DIR, f'y_{name}.npy'))
    print(f'{name:10s} X={str(X.shape):>16s}  y={str(y.shape):>10s}  '
          f'anomalies={int(y.sum()):>5,}  NaNs={int(np.isnan(X).sum())}  '
          f'range=[{X.min():.3f}, {X.max():.3f}]')

=== FINAL VERIFICATION ===



cc1_train  X=    (154198, 26)  y= (154198,)  anomalies=    0  NaNs=0  range=[-18.393, 35.106]
cc1_val    X=     (21573, 26)  y=  (21573,)  anomalies=    0  NaNs=0  range=[-16.517, 14.525]
cc1_test   X=     (44185, 26)  y=  (44185,)  anomalies=  256  NaNs=0  range=[-35.452, 22.612]
drift_sc1  X=     (59697, 26)  y=  (59697,)  anomalies=  134  NaNs=0  range=[-31.480, 26.227]
drift_sc2  X=     (38070, 26)  y=  (38070,)  anomalies=   85  NaNs=0  range=[-34.597, 31.011]
drift_cc2  X=     (76977, 26)  y=  (76977,)  anomalies=  720  NaNs=0  range=[-28.113, 26.658]


## Output files

| File | Shape | Purpose |
|---|---|---|
| `X_cc1_train.npy` / `y_...` / `ft_...` | (N, n_pc) | VAE training input — all normal |
| `X_cc1_val.npy` | (N, n_pc) | early stopping (all normal) |
| `X_cc1_test.npy` | (N, n_pc) | in-distribution eval (normal + anomaly) |
| `X_drift_sc1.npy` / `drift_sc2` / `drift_cc2` | (N, n_pc) | drift-evaluation sets, reported separately |
| `models/cc1_pca.pkl` | — | fitted PCA + window config — reuse downstream, never refit |

**Next:** `train_vae.ipynb` — loads `X_cc1_train.npy` / `X_cc1_val.npy`, defines and
trains the VAE, saves the model + training curves.